In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

**Loading the training and testing data**

In [ ]:
# Load training and testing data into DataFrames
train_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test_df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

# Display the first 5 rows of train_df
train_df.head()

**EDA**

In [ ]:
# Exploratory Data Analysis (EDA)

import matplotlib.pyplot as plt
import seaborn as sns

# 1. Basic structure and statistics
print("=== train_df.info() ===")
train_df.info()
print("\n=== train_df.describe() ===")
print(train_df.describe())

# 2. Histogram of SalePrice (raw and log-transformed)
plt.figure(figsize=(14,6))

plt.subplot(1,2,1)
sns.histplot(train_df['SalePrice'], kde=True, bins=30)
plt.title("Histogram of SalePrice (Raw)")
plt.xlabel("SalePrice")
plt.ylabel("Frequency")

plt.subplot(1,2,2)
sns.histplot(np.log1p(train_df['SalePrice']), kde=True, bins=30, color='orange')
plt.title("Histogram of SalePrice (Log-Transformed)")
plt.xlabel("Log(SalePrice + 1)")
plt.ylabel("Frequency")

plt.tight_layout()
plt.show()

# 3. Heatmap of correlations (top 10 numeric features related to SalePrice)
corr_matrix = train_df.corr(numeric_only=True)
top_corr_features = corr_matrix['SalePrice'].abs().sort_values(ascending=False).head(11).index
plt.figure(figsize=(10,8))
sns.heatmap(train_df[top_corr_features].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Heatmap of Top 10 Features Correlated with SalePrice")
plt.show()

# 4. Boxplots for categorical variables (Neighborhood vs SalePrice)
plt.figure(figsize=(16,8))
sns.boxplot(x='Neighborhood', y='SalePrice', data=train_df)
plt.xticks(rotation=45)
plt.title("SalePrice Distribution by Neighborhood")
plt.xlabel("Neighborhood")
plt.ylabel("SalePrice")
plt.show()


**Splitting the data**

In [ ]:
# Split data into features (X) and target (y)
from sklearn.model_selection import train_test_split

X = train_df.drop("SalePrice", axis=1)
y = train_df["SalePrice"]

# Perform 80/20 train-test split
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training set shape:", X_train.shape)
print("Validation set shape:", X_valid.shape)


**Calculating r^2, MSE, and RMSE**

In [ ]:
# Model training example (Linear Regression)
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Fit a simple Linear Regression model
linreg = LinearRegression()
linreg.fit(X_train.select_dtypes(include=[np.number]).fillna(0), y_train)

# Predict on validation set
y_pred = linreg.predict(X_valid.select_dtypes(include=[np.number]).fillna(0))

# Evaluation metrics
r2 = r2_score(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

print(f"R^2 Score: {r2:.4f}")
print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f}")


**Feature Engineering**

In [ ]:
# More advanced feature engineering
train_fe2 = train_df.copy()

# Interaction: Quality x TotalSF
train_fe2["TotalSF"] = (
    train_fe2["TotalBsmtSF"].fillna(0)
    + train_fe2["1stFlrSF"].fillna(0)
    + train_fe2["2ndFlrSF"].fillna(0)
)
train_fe2["Quality_SF"] = train_fe2["OverallQual"] * train_fe2["TotalSF"]

# Garage + Porch
train_fe2["Garage_Porch"] = (
    train_fe2["GarageArea"].fillna(0) 
    + train_fe2["OpenPorchSF"].fillna(0) 
    + train_fe2["EnclosedPorch"].fillna(0) 
    + train_fe2["ScreenPorch"].fillna(0) 
    + train_fe2["3SsnPorch"].fillna(0)
)

# Encode selected categorical quality features
qual_map = {"Ex":5, "Gd":4, "TA":3, "Fa":2, "Po":1}
for col in ["ExterQual", "KitchenQual"]:
    train_fe2[col + "_num"] = train_fe2[col].map(qual_map).fillna(3)  # default to TA=3

# Split into X and y
X = train_fe2.drop("SalePrice", axis=1)
y = train_fe2["SalePrice"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Retrain model
linreg = LinearRegression()
linreg.fit(X_train.select_dtypes(include=[np.number]).fillna(0), y_train)

# Predictions
y_pred = linreg.predict(X_valid.select_dtypes(include=[np.number]).fillna(0))

# Metrics
r2 = r2_score(y_valid, y_pred)
mse = mean_squared_error(y_valid, y_pred)
rmse = np.sqrt(mse)

print("With interaction and encoded quality features:")
print(f"R^2 Score: {r2:.4f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")


In [ ]:
# Train and compare models
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

models = {
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
    "GradientBoosting": GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42),
    "XGBoost": XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42, verbosity=0)
}

X_train_num = X_train.select_dtypes(include=[np.number]).fillna(0)
X_valid_num = X_valid.select_dtypes(include=[np.number]).fillna(0)

r2_scores = {}
rmse_scores = {}

for name, model in models.items():
    # Train model
    model.fit(X_train_num, y_train)
    
    # Predict on validation set
    y_pred = model.predict(X_valid_num)
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_valid, y_pred))
    rmse_scores[name] = rmse
    print(f"{name} Validation RMSE: {rmse:.2f}")
    
    # Calculate R^2
    r2 = r2_score(y_valid, y_pred)
    r2_scores[name] = r2
    print(f"{name} R^2: {r2:.4f}\n")

# Model with highest R^2
highest_r2_model = max(r2_scores, key=r2_scores.get)
print(f"Model with the highest R^2: {highest_r2_model} ({r2_scores[highest_r2_model]:.4f})")

# Model with lowest RMSE
lowest_rmse_model = min(rmse_scores, key=rmse_scores.get)
print(f"Model with the lowest RMSE: {lowest_rmse_model} ({rmse_scores[lowest_rmse_model]:.2f})")


In [ ]:
# Apply the same feature engineering to test_df
test_fe = test_df.copy()

# Total square footage
test_fe["TotalSF"] = (
    test_fe["TotalBsmtSF"].fillna(0)
    + test_fe["1stFlrSF"].fillna(0)
    + test_fe["2ndFlrSF"].fillna(0)
)

# Interaction: OverallQual * TotalSF
test_fe["Quality_SF"] = test_fe["OverallQual"] * test_fe["TotalSF"]

# Garage + Porch features
test_fe["Garage_Porch"] = (
    test_fe["GarageArea"].fillna(0)
    + test_fe["OpenPorchSF"].fillna(0)
    + test_fe["EnclosedPorch"].fillna(0)
    + test_fe["ScreenPorch"].fillna(0)
    + test_fe["3SsnPorch"].fillna(0)
)

# Encode categorical quality features
qual_map = {"Ex":5, "Gd":4, "TA":3, "Fa":2, "Po":1}
for col in ["ExterQual", "KitchenQual"]:
    test_fe[col + "_num"] = test_fe[col].map(qual_map).fillna(3)

# HouseAge
test_fe["HouseAge"] = test_fe["YrSold"] - test_fe["YearBuilt"]

# TotalBath
test_fe["TotalBath"] = (
    test_fe["FullBath"].fillna(0)
    + 0.5 * test_fe["HalfBath"].fillna(0)
    + test_fe["BsmtFullBath"].fillna(0)
    + 0.5 * test_fe["BsmtHalfBath"].fillna(0)
)

# Prepare numeric test features
X_test_num = test_fe[X_train_num.columns].fillna(0)

# Retrain the best model on full training data
best_model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
best_model.fit(X_train_num, y_train)

# Predict on test set
test_pred = best_model.predict(X_test_num)

# Create submission DataFrame
submission = pd.DataFrame({
    "Id": test_fe["Id"],
    "SalePrice": test_pred
})

# Save to CSV
submission.to_csv("submission.csv", index=False)
print("submission.csv created successfully!")
